This notebook mostly contains generalizations to the model and the optimization process defined in the CNN notebook.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# model with option number of output channels in the first layer
class NetWidth(nn.Module):
    def __init__(self, n_chans1=32):
        super().__init__()
        self.n_chans1 = n_chans1
        self.conv1 = nn.Conv2d(3,n_chans1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(8 * 8 * n_chans1 // 2, 32)
        self.fc2 = nn.Linear(32,2)

    def forward(self, x):
        out = F.max_pool2d(F.tanh(self.conv1(x)),2)
        out = F.max_pool2d(F.tanh(self.conv1(out)),2)
        out = out.view(-1, 8 * 8 * self.n_chans1 // 2)
        out = F.tanh(self.fc1(out))
        out = self.fc2(out)
        return out

In [ ]:
model = NetWidth()
sum(p.numel() for p in model.parameters())

In [ ]:
# L2 regularization (weight decay because gradient of L2 reduces weight value proportionally to current value)
import datetime

def training_loop(n_epochs, optimizer, model, loss_fn, train_loader, device):
    
    for epoch in range(1,n_epochs+1):
        loss_train = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device = device)
            labels = labels.to(device=device)
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)

            l2_lambda = 0.001 # L2 hyperparameter
            l2_norm = sum(p.pow(2.0) for p in model.parameters())
            loss = loss + l2_lambda*l2_norm

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_train += loss.item()

        if epoch == 1 or epoch % 10 == 0:
            print("{} Epoch {}, Training loss {}".format(datetime.datetime.now,
                                                         epoch,
                                                         loss_train / len(train_loader)))

In [ ]:
# Implementing the Dropout Technique from Hinton's group's paper
# "Dropout: a Simple Way to Prevent Neural Networks from Overfitting"
# The idea is to zero out a random fraction of outputs from neurons
# across the network.
# It is like augmenting the input image in a particular way that 
# aligns with the model architecture rather than us fiddling with 
# the images directly. 

class NetDropout(nn.Module):
    def __init__(self, n_chans1=32):
        super.__init__()
        self.n_chans1 = n_chans1
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
        # specify 2D specific dropout of convolution layer outputs with 
        # probability of an specific channel being zeroed
        self.conv1_dropout = nn.Dropout2d(p=0.4)
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 //2, kernel_size=3, padding=1)
        self.conv2_dropout = nn.Dropout2d(p=0.4)
        self.fc1 = nn.Linear(8 * 8 * n_chans1 // 2, 32)
        self.fc2 = nn.Linear(32,2)

    def forward(self, x):
        out = F.max_pool2d(F.tanh(self.conv1(x)),2) 
        # recall that maxpool will "shrink" the output "images"
        # but the number of channels will still be the specified number (n_chans1)
        out = self.conv1_dropout(out)
        out = F.max_pool2d(F.tanh(self.conv2(out)),2)
        out = self.conv2_dropout(out)
        out = out.view(-1,8 * 8 * self.n_chans1 // 2 )
        out = F.tanh(self.fc1(out))
        out = self.fc2(out)
        return out

Note that, with a layer like Dropout, we really need to take advantage of the context managers or settings on the model. When we evaluate the model we do not want to have Dropout applied, so we can use

model.eval()

while when for training we can set it to train mode via

model.train()

In [ ]:
# Alternatively to Dropout is Batch Normalization. The idea, from Google's paper 
# "Batch Normalization: Accelerating Deep Network Training by Reducing Internal
# Covariate Shift" is to re-normalize the outputs of layers so that we are effectively
# maximizing the gradients such a layer would yeild for the next layer's parameters.
# In other words, if the output of a layer is very "spread out" then all the values
# will be near the zero-gradient region of their activation function. Bringing them
# inwards via re-normalization would yield strong gradients, all proportional to how
# varied the layer's outputs are (i.e more extreme values stil update slower).

class NetBatchNorm(nn.Module):
    def __init__(self, n_chans1=32):
        super.__init__()
        self.n_chans1 = n_chans1
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
        self.conv1_batchnorm = nn.BatchNorm2d(num_features=n_chans1)
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3, padding=1)
        self.conv2_batchnorm = nn.BatchNorm2d(num_features= n_chans1 // 2)
        self.fc1 = nn.Linear(8 * 8 * n_chans1 // 2, 32)
        self.fc2 = nn.Linear(32,2)

    def forward(self, x):
        out = self.conv1_batchnorm(self.conv1(x)) # normalize inputs to neurons in first layer 
        out = F.max_pool2d(F.tanh(out),2)
        out = self.conv2_batchnorm(self.conv2(out)) # normalize inputs to neurons in second layer 
        out = F.max_pool2d(F.tanh(out),2)
        out = out.view(-1, 8 * 8 * self.n_chans1 // 2)
        out = F.tanh(self.fc1(out))
        out = self.fc2(out)
        return out    



As before, we need to invoke the 

model.eval()

setting when we make predictions to avoid any batch normalization. However, the model still uses the batch normalization. What is happening behind the scenes is that the layer is accumulating means and variances of the batches to get an estimated mean and variance for the "true" distribution of those values. The evaluation setting just freezes the estimator and uses the latest value. The training setting unfreezes the estimator during each forward pass. 

In [ ]:
# Here we are coding in a deep class and then will code up a ResNet where we skip certain layers to allow training
import torch 

class NetDepth(nn.Module):
    def __init__(self, n_chans1=32):
        super().__init__()
        self.n_chans1 = n_chans1
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(n_chans1 // 2, n_chans1 // 2, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(4 * 4 * n_chans1 // 2, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = F.max_pool2d(torch.relu(self.conv1(x)),2) # 32x32 image input, 16x16 "image" output x n_chans1
        out = F.max_pool2d(torch.relu(self.conv2(out)),2) # 16x16 "image" input, 8x8 "image" output x n_chans1 // 2
        out = F.max_pool2d(torch.relu(self.conv3(out)),2) # 8x8 "image" input, 4x4 "image" output x n_chans1 // 2
        # reshape data to vector
        out = out.view(-1,4*4*self.n_chans1 //2)
        out = torch.relu(self.fc1(out))
        out = self.fc2(out)
        return out

In [ ]:
class ResNet(nn.Module):
    def __init__(self, n_chans1=32):
        super().__init__()
        self.n_chans1 = n_chans1
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(n_chans1, n_chans1 // 2, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(n_chans1 // 2, n_chans1 // 2, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(4*4*n_chans1 // 2, 32)
        self.fc2 = nn.Linear(32,2)

    def forward(self, x):
        out = F.max_pool2d(torch.relu(self.conv1(x)),2)
        out = F.max_pool2d(torch.relu(self.conv2(out)),2)
        # here define the skip connection by pushing the output of
        # the above to the next layer AND a layer ahead
        out1 = out
        out = F.max_pool2d(torch.relu(self.conv3(out)) + out1, 2)
        out = out.view(-1, 4 * 4 * self.n_chans1 // 2)
        out = torch.relu(self.fc1(out))
        out = self.fc2(out)
        return out

In [ ]:
# building a VERY deep network (relative to what we have done already)

# first, define the Res Block that we will repeat many times

class ResBlock(nn.Module):
    def __init__(self, n_chans):
        # this notation may be a python2 hangover. Inserting ResBlock
        # seemingly recursively just ensures that in the Metho Resolution Order
        # the methods should start looking for whatever is above ResBlock, i.e.
        # default to the nn.Module methods when there is a conflict in method names
        super(ResBlock,self).__init__()
        # we build the 2D ConvNN, but ignore bias because Batch Norm effectively cancels its effect
        self.conv = nn.Conv2d(n_chans, n_chans, kernel_size=3, padding=1, bias=False)
        self.batch_norm = nn.BatchNorm2d(num_features=n_chans)
        # initializes the weights for the Conv2d according to a method in the original ResNet paper
        torch.nn.init.kaiming_normal_(self.conv.weight, nonlinearity='relu')
        torch.nn.init.constant_(self.batch_norm.weight, 0.5)
        torch.nn.init.zeros_(self.batch_norm.bias)

    def forward(self, x):
        out = self.conv(x)
        out = self.batch_norm(out)
        out = torch.relu(out)
        # encode in the return the skipped connection
        return out + x

In [ ]:
class NetResDeep(nn.Module):
    def __init__(self, n_chans1=32, n_blocks=10):
        super().__init__()
        self.n_chans1 = n_chans1
        self.conv1 = nn.Conv2d(3, n_chans1, kernel_size=3, padding=1)
        '''
        The book suggests the following code
        self.resblocks = nn.Sequential(
            *(n_blocks * [ResBlock(n_chans=n_chans1)] )
        )
        ChatGPT claims that this will generate the same ResBlock instance.
        A quick test suggests that is not the case.
        The initial *( ) is the "argument unpacking syntax"
        and the rest defines a list to be unpacked.
        '''
        # ChatGPT claims the following creates a new instance each time
        self.resblocks = nn.Sequential(
            *(ResBlock(n_chans=n_chans1) for _ in range(n_blocks))
            )
        # note that the ResBlocks do not maxpool, meaning we start with 
        # 32x32 tensors, the number of channels moves from 3 to n_chans.
        # The first conv1 will be maxpooled to generate 16x16 tensors,
        # but at the end of the sequence of ResBlocks, we'll downsample
        # via maxpooling to 8x8 blocks. 
        self.fc1 = nn.Linear(8 * 8 * n_chans1,32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        out = F.max_pool2d(torch.relu(self.conv1(x)),2)
        out = self.resblocks(out)
        out = F.max_pool2d(torch.relu(out),2) # there still is a relu between each res block
        out = torch.relu(self.fc1(out))
        out = self.fc2(out)
        return out
